# Sentiment de reseñas · qué arreglar primero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolenki7/Customer-Sentiment-Analysis-Product-Improvement-Drivers/blob/main/Sentiment_Analysis_and_Satisfaction_Drivers_in_E_commerce_Reviews.ipynb)

## La pregunta de negocio

Un catálogo con **100.000 reseñas** no se puede leer a mano, pero sí se puede priorizar. La pregunta no es *qué opinan los clientes* sino **qué productos hay que arreglar primero y qué señal hay que creerse**.

Este cuaderno responde:

1. **Cómo se reparte** el sentimiento en el catálogo completo.
2. **Hasta qué punto coincide** con la nota de estrellas: cuando el texto y la nota se contradicen, uno de los dos miente.
3. **Qué productos** concentran el rechazo con volumen suficiente para actuar.

**Datos.** Amazon Fine Food Reviews, 100.000 reseñas de 12.560 productos, ya puntuadas con **VADER** (`compound_score` de -1 a 1). El CSV está versionado en el repo, así que no hay que descargar nada.


In [ ]:
import pathlib
import sys

import pandas as pd

# El generador de SVG vive en src/. En Colab el repo no está clonado.
try:
    from src.svg_chart import AZUL, ROJO, barras
except ImportError:
    import urllib.request
    url = ('https://raw.githubusercontent.com/Nicolenki7/Customer-Sentiment-'
           'Analysis-Product-Improvement-Drivers/main/src/svg_chart.py')
    exec(urllib.request.urlopen(url).read().decode())

d = pd.read_csv('reviews_sentiment_TRANSFORMADO_ROBUSTO_100k.csv')
print(f'{len(d):,} reseñas · {d.product_id.nunique():,} productos')
print(d.head(3).to_string())


## 1. El censo: casi todo es positivo

Antes de buscar problemas conviene saber cuál es la línea base. Si el 90 % de las reseñas son positivas, un 8 % de negativas no es ruido: es el universo entero de queja del catálogo.


In [ ]:
cuenta = d['sentiment_category'].value_counts()
neg_pct = cuenta.get('Negative', 0) / len(d)
print((cuenta / len(d) * 100).round(1).to_string())
print(f'negativas: {neg_pct:.1%}')
print()
print('distribución de estrellas:')
print((d.rating.value_counts().sort_index() / len(d) * 100).round(1).to_string())

barras(
    [(k, v / len(d) * 100, ROJO if k == 'Negative' else AZUL)
     for k, v in cuenta.reindex(['Positive', 'Neutral', 'Negative']).items()],
    'Nueve de cada diez reseñas son positivas',
    'docs/img/sent-mix.svg',
    subtitulo=f'{len(d):,} reseñas de {d.product_id.nunique():,} productos · % del total',
    formato='{:.1f}', sufijo='%')


## 2. El hallazgo incómodo: el texto no sigue a la nota

Si el sentimiento midiera la satisfacción, las reseñas de 1 estrella serían negativas. No lo son: **casi seis de cada diez tienen sentimiento positivo**.

La explicación está en qué mide cada cosa. VADER puntúa el **tono del texto**; la estrella puntúa la **satisfacción con la compra**. Una reseña de 1 estrella típica de alimentación dice *"el producto está buenísimo pero llegó abierto y caducado"*: las palabras positivas dominan el texto aunque la experiencia sea un desastre. Por eso la media de compound **nunca llega a ser negativa**, ni siquiera en las de una estrella.

Conclusión práctica: el sentimiento **no puede sustituir** a la nota, y la nota tampoco explica el texto. Se necesitan las dos.


In [ ]:
por_nota = d.groupby('rating')['sentiment_category'].agg(
    positivas=lambda s: (s == 'Positive').mean(), n='size')
print((por_nota['positivas'] * 100).round(1).to_string())
print()
print('compound medio por nota:')
print(d.groupby('rating')['compound_score'].mean().round(3).to_string())

barras(
    [(f'{int(r)} estrella' + ('s' if r > 1 else ''), v * 100, ROJO if r <= 2 else AZUL)
     for r, v in por_nota['positivas'].items()],
    'El texto no sigue a la nota',
    'docs/img/sent-conflicto.svg',
    subtitulo='% de reseñas con sentimiento positivo, por nota de la reseña',
    formato='{:.1f}', sufijo='%')


## 3. Qué productos arreglar primero

Ordenar por porcentaje de reseñas negativas sin filtrar por volumen lleva a productos con dos reseñas y una mala. Se exige un mínimo de **20 reseñas** para que el porcentaje signifique algo: quedan 899 productos de 12.560 candidatos.


In [ ]:
por_producto = d.groupby('product_id').agg(
    n=('Id', 'size'),
    negativas=('sentiment_category', lambda s: (s == 'Negative').mean()))
con_volumen = por_producto[por_producto['n'] >= 20].sort_values('negativas', ascending=False)
peores = con_volumen.head(8)
print(f'{len(por_producto):,} productos · {len(con_volumen)} con 20+ reseñas')
print(peores.round(3).to_string())

barras(
    [(i, v * 100, ROJO) for i, v in peores['negativas'].items()],
    'Productos donde el rechazo es la norma',
    'docs/img/sent-productos.svg',
    subtitulo=f'% de reseñas negativas · 20+ reseñas · media global {neg_pct:.1%}',
    formato='{:.1f}', sufijo='%')

pathlib.Path('outputs').mkdir(exist_ok=True)
con_volumen.head(50).to_csv('outputs/worst_products.csv')


## Qué dicen los números

- **89,9 % de reseñas positivas**, 8,0 % negativas y 2,2 % neutras sobre 100.000 reseñas. La queja real del catálogo son **7.987 reseñas**.
- **El conflicto es la historia:** 59,1 % de las reseñas de 1 estrella tienen sentimiento positivo (5.506 de 9.318), y sube a 74,7 % en las de 2. En el otro extremo, solo el 3,3 % de las de 4-5 estrellas son negativas.
- **El compound medio por nota va de 0,184 a 0,777**: nunca es negativo, ni en las de una estrella. Un modelo que solo mirara el texto jamás detectaría un producto roto.
- **El rechazo se concentra:** con el 8,0 % de media global, un producto llega al **65,8 %** de reseñas negativas (B00002N8SM, 38 reseñas). Un puñado de SKUs explica una parte desproporcionada del problema.

### Limitaciones

- **VADER es un léxico, no un modelo entrenado**: no entiende negación compleja ni sarcasmo, y sobre reseñas de comida se sesga a positivo porque el vocabulario gastronómico es elogioso. El 59 % de conflicto en 1 estrella es en parte eso.
- El CSV no trae el texto original, solo las puntuaciones ya calculadas: aquí **no se puede auditar** qué palabras dispararon cada score.
- El corte de 20 reseñas es una decisión de negocio, no estadística; con 899 productos por encima, cambiar el umbral cambia el ranking.
- No hay fecha ni categoría de producto: no se puede ver si el rechazo empeoró con el tiempo ni separar alimentación de otros departamentos.

### Siguiente paso

Cambiar el léxico por un modelo entrenado sobre estas mismas notas (o por un LLM con pocas etiquetas) y comprobar si el conflicto baja. Si baja, el problema era la herramienta; si no baja, el problema es que la gente puntúa la experiencia logística y escribe sobre el sabor — y entonces la prioridad de negocio no está en el producto.
